**The purpose of this notebook:** apply the clean function and attend the specific needs of the dataset to The final cleaned dataset to be used for further analysis and modeling.


In [1]:
import pandas as pd
import yaml
import openpyxl

---

In [2]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config").is_dir())
with open(ROOT / "config" / "config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
cfg = {k: str(ROOT / v) for k, v in cfg.items()}  # relative paths resolve against the project root

In [4]:
P_A2_24 = pd.read_excel(cfg["P_A2_24_path"], engine="openpyxl")

---

In [5]:
import sys
sys.path.append("../..")
from src.functions import clean_plant

In [6]:
P_A2_24_clean = clean_plant(P_A2_24)
P_A2_24_clean

Rows with all NaN values: 1408
Rows with 'Dosif. m3' in first column: 1409


,\n \n Fecha,\n \n Fórmula,\n \n Volumen,"Cemento\n 1\n CEM I 42,5 R",Cemento\n 4\n CEM IV/A(V),Arido\n 1\n ARENA 04,Arido\n 2\n GRAVA 8/14,Arido\n 3\n GRAVA 12/20,Agua\n 1\n AGUA,Aditivo\n 2\n DYNAMON SX-3,Aditivo\n 4\n COMPLAST MR-,Aditivo\n 5\n AUT DINAMON
0,2024-12-30,AUTO 300 CT-C12-F3,6.5,1735.0,0.0,10920.0,0.0,0.0,1429.0,0.0,0.0,17.90
1,2024-12-30,AUTO 300 CT-C12-F3,5.0,1256.0,0.0,8560.0,0.0,0.0,1098.0,0.0,0.0,13.80
2,2024-12-30,AUTO 300 CT-C12-F3,4.5,1129.0,0.0,7480.0,0.0,0.0,987.0,0.0,0.0,12.40
3,2024-12-30,AUTO 300 CT-C12-F3,8.0,2028.0,0.0,13650.0,0.0,0.0,1755.0,0.0,0.0,22.00
4,2024-12-30,AUTO 300 CT-C12-F3,6.0,1508.0,0.0,9990.0,0.0,0.0,1319.0,0.0,0.0,16.50
...,...,...,...,...,...,...,...,...,...,...,...,...
1403,2024-12-20,RES.H-250/B/20,7.0,1745.0,0.0,7900.0,2270.0,4180.0,917.0,8.7,11.3,0.00
1404,2024-12-20,RES.H-250/B/20,7.0,1744.0,0.0,7680.0,2260.0,4200.0,1035.0,8.7,11.4,0.00
1405,2024-11-19,PILARES AÑIL HA-30/F/20/X0,1.0,79.0,0.0,0.0,126.0,0.0,5.0,0.0,2.8,0.00
1406,2024-11-19,AUTO 300 CT-C12-F3,1.0,252.0,0.0,1700.0,0.0,0.0,220.0,0.0,0.0,3.28


---

In [7]:
# How many non-zero values each column has and what percentage of all rows that represents.
non_zero_counts = (P_A2_24_clean != 0.0).sum()
non_zero_pct = non_zero_counts / len(P_A2_24_clean) * 100

pd.DataFrame({
    "non_zero_count": non_zero_counts,
    "non_zero_pct": non_zero_pct.round(2)
})

,non_zero_count,non_zero_pct
\n \n Fecha,1408,100.00
\n \n Fórmula,1408,100.00
\n \n Volumen,1408,100.00
"Cemento\n 1\n CEM I 42,5 R",141,10.01
Cemento\n 4\n CEM IV/A(V),1243,88.28
Arido\n 1\n ARENA 04,1387,98.51
Arido\n 2\n GRAVA 8/14,464,32.95
Arido\n 3\n GRAVA 12/20,405,28.76
Agua\n 1\n AGUA,1384,98.30
Aditivo\n 2\n DYNAMON SX-3,481,34.16


In [8]:
P_A2_24_clean.shape

(1408, 12)

In [9]:
P_A2_24_clean.columns = ["date", "product", "volume", "cement_42_5r", "cement_4av", "arid_04", "arid_8_14", "arid_12_20","water","ad_dynamon_sx","ad_complast_mr", "ad_aut_dynamon"]

In [10]:
P_A2_24_clean.columns.tolist()

['date',
 'product',
 'volume',
 'cement_42_5r',
 'cement_4av',
 'arid_04',
 'arid_8_14',
 'arid_12_20',
 'water',
 'ad_dynamon_sx',
 'ad_complast_mr',
 'ad_aut_dynamon']

In [11]:
# Frequency of rows where any of two or more of the following products; "cement_42_5r", "cement_3a_42", "cement_4a_av","cement_4a_av_1" are !=0 in the same row.
# Frequency of rows where any 2 of the cement products are !=0 in the same row.
cement_columns = ["cement_42_5r", "cement_4av"]

# Count non-zero values per row for these columns.
non_zero_cement_count = (P_A2_24_clean[cement_columns] != 0).sum(axis=1)

# Find rows where 2 cement products are non-zero.
condition = non_zero_cement_count == 2
frequency = condition.sum()

print(f"Frequency of rows where 2 cement products are non-zero: {frequency}")
print(f"Percentage: {(frequency / len(P_A2_24_clean) * 100):.2f}%")

Frequency of rows where 2 cement products are non-zero: 0
Percentage: 0.00%


In [12]:
## we add a new column where all values will be P_A2 because in the future merging we need to identify from which plant the load is come from. 
# Inserting a new column and value. 
# P_A2_24_clean["plant"] = "P_A2"
# Inserting a column position too. 
P_A2_24_clean.insert(1, "plant", "P_A2")

In [13]:
P_A2_24_clean.columns.tolist()

['date',
 'plant',
 'product',
 'volume',
 'cement_42_5r',
 'cement_4av',
 'arid_04',
 'arid_8_14',
 'arid_12_20',
 'water',
 'ad_dynamon_sx',
 'ad_complast_mr',
 'ad_aut_dynamon']

In [14]:
# The final datsets (all plants and both years merged) will have the following columns in this order:
# ['date', 'plant', 'product', 'volume', 'cement_42_5r', 'cement_3a_42', 'cement_4av', 'arid_04', 'arid_8_14', 'arid_12_20', 'water', 'ad_complast_mr', 'ad_dynamon_sx', 'ad_aut_dynamon', 'ad_auramix', 'corrective_02']
# This step could be done in the merge file, but the column name need to be the same in all plants and years, so we do it here. So instead of doing just this step we order them too.
final_order = [
    'date',
    'plant',
    'product',
    'volume',
    'cement_42_5r',
    'cement_4av',
    'arid_04',
    'arid_8_14',
    'arid_12_20',
    'water',
    'ad_complast_mr',
    'ad_dynamon_sx',
    'ad_aut_dynamon',
]

P_A2_24_clean = P_A2_24_clean[final_order]
print(P_A2_24_clean.columns.tolist())

['date', 'plant', 'product', 'volume', 'cement_42_5r', 'cement_4av', 'arid_04', 'arid_8_14', 'arid_12_20', 'water', 'ad_complast_mr', 'ad_dynamon_sx', 'ad_aut_dynamon']


---

In [15]:
P_A2_24_clean.to_csv(cfg["P_A2_24_ready_csv_path"], index=False, encoding="utf-8-sig")